# Кейс 21. Рекомендации ещё не сыгранных героев

Цель: по implicit feedback игрок-герой обучить ALS и выдать top-5 героев, которыми игрок ещё не играл. Путь: источник → учебный RAW → сверка → STAGE → MART → качество/Iceberg → потребление.

## Методика, алгоритм и метрики

Это рекомендательная задача, а не задача для языковой модели. Под «неявной обратной связью» понимается отсутствие прямой оценки героя: сам факт матчей и побед косвенно показывает интерес игрока. На этапе STAGE строится одна строка для каждой пары `account_id, hero_id`, а `implicit_strength` задаёт силу такого взаимодействия для ALS.

Алгоритм ALS (чередующихся наименьших квадратов) раскладывает матрицу «игрок–герой» на скрытые векторы игроков и героев. Параметр `rank=12` задаёт размер скрытого пространства, `maxIter=12` — число итераций, `regParam=0.1` — силу регуляризации, `alpha=1.0` — вес уверенности в наблюдаемом взаимодействии. Метод `recommendForAllUsers` создаёт кандидатов, после чего уже сыгранные герои удаляются операцией `left_anti`.

Проверяются уникальность пары «игрок–герой», диапазон рангов от 1 до 5 и отсутствие ранее сыгранных героев. Для полноценной проверки качества нужны `precision@5` (доля релевантных рекомендаций среди пяти), `recall@5` (доля найденных релевантных героев) и `MAP@5` или `NDCG@5` на отложенных матчах. Текущий notebook проверяет только структуру результата и не проводит проверку на будущих матчах. ALS также не решает проблему новых игроков без истории.

## 1. Источники

Используются PostgreSQL `dota.match_players` и `dota.matches`. Grain источника игроков — `(match_id, player_slot)`. Анонимный account `4294967295` не подходит для персонализации.

In [ ]:
import os
import psycopg2

pg_dsn = os.getenv("SOURCE_POSTGRES_DSN")
if not pg_dsn:
    raise RuntimeError("Нужен SOURCE_POSTGRES_DSN")
    
with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        cursor.execute("""
            select count(*), count(distinct account_id), count(*) 
                filter (where account_id is null or account_id = 4294967295), count(*) 
                filter (where hero_id is null) 
            from dota.match_players
        """)
        print(dict(zip(["rows", "accounts", "anonymous_rows", "null_hero"], cursor.fetchone())))
        cursor.execute("select count(*), min(start_time), max(start_time) from dota.matches")
        print("matches:", cursor.fetchone())

## 2. Учебный RAW

Читаем источники JDBC и сохраняем отдельные Iceberg-таблицы. RAW не фильтрует анонимов: очистка относится к STAGE.

In [ ]:
from psycopg2.extensions import parse_dsn
from pyspark.sql import functions as F
from lakehouse_spark import SparkSessionContext
from lakehouse_publish import publish_table

pg = parse_dsn(pg_dsn)
jdbc_url = f"jdbc:postgresql://{pg['host']}:{pg.get('port', '5432')}/{pg['dbname']}"
jdbc = {"user": pg["user"], "password": pg["password"], "driver": "org.postgresql.Driver"}

with SparkSessionContext("practice-case-21-raw") as ctx:
    spark = ctx
    
    players = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dota.match_players")
            .options(**jdbc)
            .load()
            .select("match_id", "player_slot", "account_id", "hero_id", "win", "loaded_at")
            .withColumnRenamed("loaded_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    matches = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dota.matches")
            .options(**jdbc)
            .load()
            .select("match_id", "start_time", "stratz_parsed_at")
            .withColumnRenamed("stratz_parsed_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    for name, frame in {"match_players": players, "matches": matches}.items():
        with ctx.lineage(
            f"practice-case-21-raw-{name}", 
            output=f"raw.practice_case_21_{name}", 
            description="Загрузка исходных данных для рекомендаций героев",
            dataset_descriptions={f"raw.practice_case_21_{name}": f"Учебная RAW-таблица {name} для кейса о рекомендациях героев."}
        ):
            frame.writeTo(f"lakehouse.raw.practice_case_21_{name}").using("iceberg").createOrReplace()
            publish_table("raw", f"practice_case_21_{name}", frame, description=f"Учебная RAW-таблица {name} для рекомендаций героев")
            
        print(name, frame.count())

## 3. Сверка RAW

Сверяем counts, grain игроков и временной диапазон матчей.

In [ ]:
with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        cursor.execute("select count(*), count(distinct (match_id, player_slot)) from dota.match_players")
        source_players = cursor.fetchone()

        cursor.execute("select count(*), min(start_time), max(start_time) from dota.matches")
        source_matches = cursor.fetchone()
        
with SparkSessionContext("practice-case-21-reconcile") as spark:
    raw_players = tuple(spark.sql("select count(*), count(distinct struct(match_id, player_slot)) from lakehouse.raw.practice_case_21_match_players").first())
    raw_matches = tuple(spark.sql("select count(*), min(start_time), max(start_time) from lakehouse.raw.practice_case_21_matches").first())
    
    for name, source_value, raw_value in [("players", source_players, raw_players), ("matches", source_matches, raw_matches)]:
        print(name, "PASS" if source_value == raw_value else "FAIL", {"source": source_value, "raw": raw_value})

## 4. STAGE: неявная обратная связь

Оставляем игроков минимум с тремя матчами и двумя разными героями. Strength = число матчей + удвоенное число побед; это положительный implicit signal, а не явная оценка.

In [ ]:
with SparkSessionContext("practice-case-21-stage") as ctx:
    spark = ctx
    with ctx.lineage(
        "practice-case-21-interactions", 
        inputs=["raw.practice_case_21_match_players", "raw.practice_case_21_matches"], 
        output="stage.practice_case_21_player_hero_interactions", 
        description="Неявная обратная связь между игроком и героем", 
        dataset_descriptions={"stage.practice_case_21_player_hero_interactions": "Агрегированные взаимодействия игрока с героем и сила неявного сигнала для рекомендаций."},
        config={"strength": "matches + 2 * wins"}
    ):
        interactions = spark.sql("""
            with eligible as (
                select account_id from lakehouse.raw.practice_case_21_match_players
                where account_id is not null and account_id <> 4294967295 and hero_id is not null
                group by account_id having count(*) >= 3 and count(distinct hero_id) >= 2
            )
            select p.account_id, p.hero_id, count(*) matches_played, sum(case when p.win then 1 else 0 end) wins,
                   cast(count(*) as double) + 2.0 * sum(case when p.win then 1 else 0 end) implicit_strength,
                   cast(max(m.start_time) as bigint) last_played_at
            from lakehouse.raw.practice_case_21_match_players p join eligible e on p.account_id = e.account_id
            join lakehouse.raw.practice_case_21_matches m on p.match_id = m.match_id
            where p.hero_id is not null group by p.account_id, p.hero_id
        """)
        interactions.writeTo("lakehouse.stage.practice_case_21_player_hero_interactions").using("iceberg").createOrReplace()
        publish_table("stage", "practice_case_21_player_hero_interactions", interactions, description="История взаимодействий игроков с героями", dependencies=["raw/practice_case_21_match_players", "raw/practice_case_21_matches"])
    interactions.show(10, truncate=False)

## 5. MART: implicit ALS

Присваиваем игрокам числовые индексы, обучаем ALS, получаем список кандидатов и удаляем уже сыгранных героев с помощью анти-соединения.

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.sql import Window, functions as F

with SparkSessionContext("practice-case-21-mart") as ctx:
    spark = ctx
    with ctx.lineage(
        "practice-case-21-als", 
        inputs=["stage.practice_case_21_player_hero_interactions"], 
        output="mart.practice_case_21_hero_recommendations", 
        description="Пять ещё не сыгранных героев, рекомендованных с помощью ALS", 
        dataset_descriptions={"mart.practice_case_21_hero_recommendations": "Персональные рекомендации пяти ещё не сыгранных героев для каждого игрока."},
        config={"rank": 12, "top_n": 5}
    ):
        source = spark.table("lakehouse.stage.practice_case_21_player_hero_interactions")
        indexer = StringIndexer(inputCol="account_id", outputCol="user_index").fit(source)
        indexed = indexer.transform(source).withColumn("user_id", F.col("user_index").cast("int"))
        
        model = ALS(
            userCol="user_id", itemCol="hero_id", ratingCol="implicit_strength", 
            implicitPrefs=True, rank=12, maxIter=12, regParam=0.1, alpha=1.0, 
            nonnegative=True, coldStartStrategy="drop", seed=42
        ).fit(indexed)
        hero_count = indexed.select("hero_id").distinct().count()
        candidates = (
            model.recommendForAllUsers(hero_count)
                .select("user_id", F.explode("recommendations").alias("rec"))
                .select("user_id", F.col("rec.hero_id").alias("hero_id"), F.col("rec.rating").alias("predicted_affinity"))
        )
        
        seen = indexed.select("user_id", "hero_id").distinct()
        users = indexed.select("user_id", "account_id").distinct()
        
        window = Window.partitionBy("user_id").orderBy(F.desc("predicted_affinity"), "hero_id")
        
        result = (
            candidates.join(seen, ["user_id", "hero_id"], "left_anti")
                .withColumn("recommendation_rank", F.row_number().over(window))
                .where("recommendation_rank <= 5")
                .join(users, "user_id")
                .select("account_id", "hero_id", "recommendation_rank", "predicted_affinity")
                .withColumn("model_version", F.lit("practice_als_implicit_v1"))
        )
        result.writeTo("lakehouse.mart.practice_case_21_hero_recommendations").using("iceberg").createOrReplace()
        publish_table("mart", "practice_case_21_hero_recommendations", result, description="Персональные рекомендации героев для игроков", dependencies=["stage/practice_case_21_player_hero_interactions"])
    
    result.orderBy("account_id", "recommendation_rank").show(20, truncate=False)

## 6. Качество, Iceberg и потребление

Проверяем уникальность взаимодействий, диапазон рангов и отсутствие сыгранных героев. ALS ранжирует героев на основе истории игрока и не решает проблему новых игроков без истории.

In [ ]:
with SparkSessionContext("practice-case-21-quality") as spark:
    checks = {
        "stage_grain": "select count(*) - count(distinct struct(account_id, hero_id)) violations from lakehouse.stage.practice_case_21_player_hero_interactions",
        "rank_range": "select count_if(recommendation_rank not between 1 and 5) violations from lakehouse.mart.practice_case_21_hero_recommendations",
        "unseen_only": "select count(*) violations from lakehouse.mart.practice_case_21_hero_recommendations r join lakehouse.stage.practice_case_21_player_hero_interactions s on r.account_id = s.account_id and r.hero_id = s.hero_id",
    }
    
    for name, query in checks.items():
        violations = spark.sql(query).first()["violations"]
        print(name, "PASS" if violations == 0 else f"FAIL: {violations}")
    spark.sql("""
        select snapshot_id, committed_at, operation 
        from lakehouse.mart.practice_case_21_hero_recommendations.snapshots 
        order by committed_at desc 
        limit 3
    """).show(truncate=False)
    
    spark.sql("""
        select account_id, hero_id, recommendation_rank, round(predicted_affinity, 4) predicted_affinity 
        from lakehouse.mart.practice_case_21_hero_recommendations 
        order by account_id, recommendation_rank 
        limit 30
    """).show(truncate=False)

## 7. Аналитика MART и гипотезы

Проверяем, какие герои чаще всего попадают в первые пять рекомендаций, и насколько равномерно рекомендации распределены между героями. Гипотеза H1: модель чаще рекомендует героев с сильным сигналом взаимодействий в обучающей истории. Гипотеза H2: рекомендации не должны состоять только из одного-двух самых популярных героев. Для полноценной проверки нужны отложенные будущие матчи и метрики точности; текущая ячейка выполняет описательный анализ MART.

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

with SparkSessionContext("practice-case-21-analytics") as spark:
    recommendations = spark.table("lakehouse.mart.practice_case_21_hero_recommendations")
    top_heroes = (
        recommendations.groupBy("hero_id")
            .agg(F.count("*").alias("recommendations"), F.round(F.avg("predicted_affinity"), 3).alias("avg_affinity"))
            .orderBy(F.desc("recommendations"), F.desc("avg_affinity"))
            .limit(15)
    )
    top_heroes.show(truncate=False)
    plot = top_heroes.toPandas().sort_values("recommendations")

plt.figure(figsize=(10, 5))
plt.barh(plot["hero_id"].astype(str), plot["recommendations"], color="#6b8e62")
plt.title("Герои, чаще всего попавшие в рекомендации")
plt.xlabel("Число рекомендаций")
plt.ylabel("Идентификатор героя")
plt.tight_layout()
plt.show()